In [ ]:
import pandas as pd
import glob
import yaml
import os
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
dataset_config = yaml.safe_load(open("config/dataset.yml"))
openalex_snapshot_dir = os.environ.get(
    "OPENALEX_CSV_CONFIG_DIR",
    os.path.join(dataset_config["path_openalex"], "csv-files_config"),
)
openalex_title_year_path = os.environ.get(
    "OPENALEX_TITLE_YEAR_PARQUET",
    os.path.join(openalex_snapshot_dir, "work_title_year.parquet"),
)

In [ ]:
from pandarallel import pandarallel
pandarallel.initialize(progress_bar=False)

## Import OA info

In [ ]:
def parallel_load(KEYWORD, folder='csv-files'):

    file_pattern = os.path.join(f'{folder}\\{KEYWORD}', f'{KEYWORD}_*.csv.gz')
    csv_files = glob.glob(file_pattern)

    def load_and_rename(path):
        try:
            df = pd.read_csv(path, compression='gzip')
            if not df.empty:
                return df.rename(columns={'work_id': 'oaid'})
        except Exception as e:
            print(f"Error reading {path}: {e}")
        return None

    dataframes = []
    with ThreadPoolExecutor(max_workers=100) as executor:
        # submit all tasks
        futures = {executor.submit(load_and_rename, f): f for f in csv_files}
        # collect results as they finish
        for future in tqdm(as_completed(futures), total=len(futures), desc="Loading files"):
            df = future.result()
            if df is not None:
                dataframes.append(df)

    merged_df = pd.concat(dataframes, ignore_index=True).drop_duplicates()
    del dataframes

    return merged_df

In [ ]:
oa_title = parallel_load('works_title', folder=openalex_snapshot_dir).rename(columns={})
oa_title

In [ ]:
oa_year = parallel_load('works_year', folder=openalex_snapshot_dir).rename(columns={})
oa_year

In [ ]:
oa_title_year = oa_title.merge(oa_year, how='left')
oa_title_year

In [ ]:
oa_title_year.to_parquet(openalex_title_year_path)